# Deploy a fine-tuned model to SageMaker real-time inference (multi-LoRA)

This notebook deploys a LoRA fine-tune produced by **Training Sessions** to a SageMaker
real-time endpoint, using the **base + adapter** architecture:

```
SageMaker endpoint (GPU instance)
└── base inference component      base model weights, vLLM, LoRA enabled
    ├── adapter IC: your run #1   ~1 GB, attaches in seconds
    ├── adapter IC: your run #2
    └── ...
```

Why this shape instead of merging weights per fine-tune:

* **Every checkpoint save is instantly deployable** — each weight snapshot's adapter
  (~1 GB) becomes an adapter component; no weight merging, no multi-GB artifact per run.
* **One endpoint serves all your fine-tunes** — new adapters attach to the running base in
  seconds; each invocation picks its adapter by name.
* **It is the same architecture your training ran on** — Training Sessions engines serve
  multi-LoRA over a shared base; the endpoint mirrors it.

The flow follows the AWS container-standards guide for
[LoRA adapter management with inference components](https://github.com/aws/model-hosting-container-standards/blob/main/docs/sagemaker/05_lora_adapter_management.md),
with values this cookbook has validated end-to-end.

The flow is model-agnostic: everything is resolved from your model package's `BaseModel`
field, so it works for any base model the cookbook trains — verified against gpt-oss,
gemma, and qwen3 packages/hub entries. Only the instance sizing knobs in Step 1 depend on
the model's size. (Endpoint flow validated end-to-end with gpt-oss-20b.)

**Cost warning:** real-time endpoints bill per instance-hour while `InService` — there is no
scale-to-zero. Run the cleanup step when you are done.


## Step 1 — Configuration

Fill in the three placeholders below:

* **`MODEL_PACKAGE_ARN`** — the fine-tuned model you want to deploy. Training runs
  register each saved checkpoint as a SageMaker **model package** in your model package
  group; the ARN is printed in the training log at every save, or copy it from the
  SageMaker console under *Models → Model packages* (pick the version you want, usually
  the latest).
* **`ROLE_ARN`** — an IAM role the endpoint runs as (your training execution role works).
  Step 3 lists the permissions it needs.
* **`STAGING_BUCKET`** — an S3 bucket in the deploy region, used to stage the base model
  weights and your adapter.

One thing to know about model packages: training produces two kinds. **Weight snapshots**
(created when the run saves weights for sampling/deployment) are deployable. **Training-state
checkpoints** (optimizer state for resuming a run) are not. Step 2 checks this automatically
and tells you if you picked the wrong kind.


In [ ]:
import boto3, json, tarfile, tempfile, time, os
from pathlib import Path

ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

# --- REQUIRED: your values -------------------------------------------------------
MODEL_PACKAGE_ARN = 'YOUR_MODEL_PACKAGE_ARN'   # arn:aws:sagemaker:...:model-package/<group>/<version>
DEPLOY_REGION     = 'us-east-1'                # region for the endpoint (see capacity note below)
ROLE_ARN          = f'arn:aws:iam::{ACCOUNT_ID}:role/YOUR_EXECUTION_ROLE'  # see Step 3 for required permissions
STAGING_BUCKET    = 'YOUR_BUCKET'              # same-region bucket for base weights + adapter tarball

# --- endpoint shape --------------------------------------------------------------
# Size for your base model: bf16 weights need ~2 GB VRAM per billion parameters, plus
# ~25% headroom for KV cache and LoRA slots. Rules of thumb for the cookbook bases:
#   ~20B (gpt-oss-20b):        4x24GB+ GPUs, TP=4  (validated: ml.g7.24xlarge; also
#                              g6e.12xlarge / g5.12xlarge / g6.12xlarge)
#   ~27B (qwen3):              4x24GB is tight; prefer 4x48GB (g6e.12xlarge), TP=4
#   smaller bases (<10B):      a single 24GB+ GPU works, TP=1
# Capacity note: multi-GPU pools run dry regionally; failures surface only after ~30 min
# as InsufficientInstanceCapacity. If that happens, try another type/region.
INSTANCE_TYPE  = 'ml.g7.24xlarge'
TENSOR_PARALLEL = '4'                          # = GPU count of the instance type

ENDPOINT_NAME = 'cookbook-lora-endpoint'
BASE_IC       = f'{ENDPOINT_NAME}-base'

# vLLM Deep Learning Container, SageMaker variant.
# Use an ubuntu-based tag: as of Sep 2026 the AL2023-based `server-sagemaker-cuda-v2.4`
# tags are missing the `which` binary and crash during engine warmup for MoE models.
VLLM_IMAGE = (f'763104351884.dkr.ecr.{DEPLOY_REGION}.amazonaws.com/'
              'vllm:0.28.0-gpu-py312-cu130-ubuntu24.04-sagemaker-v1.0-2026-08-26-17-31-59')

assert 'YOUR_' not in MODEL_PACKAGE_ARN + ROLE_ARN + STAGING_BUCKET, (
    'fill in MODEL_PACKAGE_ARN, ROLE_ARN and STAGING_BUCKET above')

sm  = boto3.client('sagemaker', region_name=DEPLOY_REGION)
s3  = boto3.client('s3', region_name=DEPLOY_REGION)
smr = boto3.client('sagemaker-runtime', region_name=DEPLOY_REGION)
train_region = MODEL_PACKAGE_ARN.split(':')[3]
sm_train = boto3.client('sagemaker', region_name=train_region)
print('account:', ACCOUNT_ID, '| deploy region:', DEPLOY_REGION, '| package region:', train_region)

## Step 2 — Resolve everything from the model package

Every Training Sessions model package carries a `BaseModel` field — the deployment linkage
the platform writes for you. From the package alone we resolve:

* which **base model and hub version** the adapter was trained against (so the served base
  is pinned to the exact training base, by construction), and
* where the **adapter weights** live (`checkpoints/hf/` under the package's data source).


In [ ]:
pkg = sm_train.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)
container = pkg['InferenceSpecification']['Containers'][0]

assert not container.get('IsCheckpoint', False), (
    'this package is a training-state checkpoint (for resuming training), not a '
    'deployable weight snapshot — pick a different version from your model package group')

base_model = container['BaseModel']          # {HubContentName, HubContentVersion, RecipeName}
package_s3 = container['ModelDataSource']['S3DataSource']['S3Uri']
ADAPTER_PREFIX = package_s3 + 'checkpoints/hf/'
print('base model :', base_model['HubContentName'], base_model['HubContentVersion'])
print('adapter    :', ADAPTER_PREFIX)

# Resolve the base model's hosting artifact (pre-packaged weights in the JumpStart cache)
hub = sm.describe_hub_content(HubName='SageMakerPublicHub', HubContentType='Model',
                              HubContentName=base_model['HubContentName'],
                              HubContentVersion=base_model['HubContentVersion'])
hub_doc = json.loads(hub['HubContentDocument'])
BASE_ARTIFACT_SRC = hub_doc['HostingArtifactUri']   # s3://jumpstart-cache-prod-<region>/...
print('base weights:', BASE_ARTIFACT_SRC)

## Step 3 — Stage the artifacts

Two copies into your staging bucket:

1. **Base weights** — the endpoint's validation reads S3 as *your* execution role, which
   cannot read the cross-account JumpStart cache, so we copy the prepacked base weights
   (size varies by model — ~13 GB for gpt-oss-20b; a server-side copy, 1–2 minutes) into
   your bucket once. All future
   deployments reuse it.
2. **Adapter as `adapter.tar.gz`** — adapter inference components require a single tarball
   (`Container.ArtifactUrl` accepts only `.tar.gz`).

Your execution role needs: S3 read on both staged prefixes, ECR pull
(`ecr:GetAuthorizationToken/BatchGetImage/GetDownloadUrlForLayer`), and CloudWatch Logs write.


In [ ]:
from urllib.parse import urlparse

def s3_copy_prefix(src_uri, dst_bucket, dst_prefix, src_region):
    src = urlparse(src_uri); s3src = boto3.client('s3', region_name=src_region)
    keys = []
    for page in s3src.get_paginator('list_objects_v2').paginate(
            Bucket=src.netloc, Prefix=src.path.lstrip('/')):
        keys += [o['Key'] for o in page.get('Contents', [])]
    for k in keys:
        rel = k[len(src.path.lstrip('/')):]
        s3.copy({'Bucket': src.netloc, 'Key': k}, dst_bucket, dst_prefix + rel,
                SourceClient=s3src)
    return len(keys)

# 1) base weights (skip if already staged)
base_name = f"base/{base_model['HubContentName']}-{base_model['HubContentVersion']}/"
BASE_ARTIFACT = f's3://{STAGING_BUCKET}/{base_name}'
if 'Contents' not in s3.list_objects_v2(Bucket=STAGING_BUCKET, Prefix=base_name, MaxKeys=1):
    n = s3_copy_prefix(BASE_ARTIFACT_SRC, STAGING_BUCKET, base_name, DEPLOY_REGION)
    print(f'staged base weights ({n} objects) ->', BASE_ARTIFACT)
else:
    print('base weights already staged ->', BASE_ARTIFACT)

# 2) adapter tar.gz
pkg_version = MODEL_PACKAGE_ARN.rsplit('/', 1)[-1]
ADAPTER_TARBALL = f's3://{STAGING_BUCKET}/adapters/v{pkg_version}/adapter.tar.gz'
with tempfile.TemporaryDirectory() as tmp:
    src = urlparse(ADAPTER_PREFIX); s3src = boto3.client('s3', region_name=train_region)
    for page in s3src.get_paginator('list_objects_v2').paginate(
            Bucket=src.netloc, Prefix=src.path.lstrip('/')):
        for o in page.get('Contents', []):
            rel = o['Key'][len(src.path.lstrip('/')):]
            s3src.download_file(src.netloc, o['Key'], os.path.join(tmp, rel))
    tar_path = os.path.join(tmp, 'adapter.tar.gz')
    with tarfile.open(tar_path, 'w:gz') as tar:
        for name in os.listdir(tmp):
            if name != 'adapter.tar.gz':
                tar.add(os.path.join(tmp, name), arcname=name)
    dst = urlparse(ADAPTER_TARBALL)
    s3.upload_file(tar_path, dst.netloc, dst.path.lstrip('/'))
print('adapter tarball ->', ADAPTER_TARBALL)

## Step 4 — Create the endpoint

An inference-component endpoint: the variant reserves the instance but names no model —
components land on it afterwards. This is the long step (instance provisioning). If the
region has no capacity for the instance type, the endpoint fails **terminally** with
`InsufficientInstanceCapacity` — there is no early signal, so we poll status.


In [ ]:
try:
    sm.create_endpoint_config(
        EndpointConfigName=ENDPOINT_NAME, ExecutionRoleArn=ROLE_ARN,
        ProductionVariants=[{
            'VariantName': 'main', 'InstanceType': INSTANCE_TYPE, 'InitialInstanceCount': 1,
            'ModelDataDownloadTimeoutInSeconds': 1200,
            'ContainerStartupHealthCheckTimeoutInSeconds': 1200,
            'RoutingConfig': {'RoutingStrategy': 'LEAST_OUTSTANDING_REQUESTS'},
        }])
    sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_NAME)
    print('endpoint create requested')
except sm.exceptions.ClientError as e:
    if 'already exist' not in str(e).lower(): raise
    print('endpoint (or config) already exists — continuing')

t0 = time.time()
while True:
    d = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    if d['EndpointStatus'] == 'InService':
        print(f'InService after {time.time()-t0:.0f}s'); break
    if d['EndpointStatus'] == 'Failed':
        raise RuntimeError(f"endpoint failed: {d.get('FailureReason')}")
    print(f"{d['EndpointStatus']} ({time.time()-t0:.0f}s)"); time.sleep(30)

## Step 5 — Base inference component

The base is defined as a SageMaker **Model** (only Model objects accept uncompressed S3
prefixes) with the vLLM image and LoRA-serving configuration:

* `SM_VLLM_*` variables become vLLM server arguments (`SM_VLLM_MAX_LORA_RANK` → `--max-lora-rank`).
* `VLLM_ALLOW_RUNTIME_LORA_UPDATING=true` (no `SM_VLLM_` prefix) is **required** — without
  it vLLM rejects the platform's adapter registration and every adapter invoke returns 404.
* `SM_VLLM_DTYPE=bfloat16` pins serving numerics to bf16 — the dtype Training Sessions
  engines compute in for every supported base. For natively-quantized bases (gpt-oss ships
  MXFP4) this forces dequantized bf16 weights, matching training exactly; without the pin,
  FP4-capable GPUs would use different kernel numerics than your adapter was trained against.

Base startup ≈ 5–8 minutes (weight pull + engine init).


In [ ]:
BASE_MODEL_NAME = f'{ENDPOINT_NAME}-base-model'
ENV = {
    'SM_VLLM_DTYPE': 'bfloat16',
    'SM_VLLM_TENSOR_PARALLEL_SIZE': TENSOR_PARALLEL,
    'SM_VLLM_MAX_MODEL_LEN': '16384',
    'SM_VLLM_GPU_MEMORY_UTILIZATION': '0.85',
    'SM_VLLM_MAX_NUM_SEQS': '8',              # keeps the LoRA warmup within VRAM
    'SM_VLLM_ENABLE_LORA': 'true',
    'SM_VLLM_MAX_LORA_RANK': '64',            # >= your adapters' rank (cookbook default: 32)
    'SM_VLLM_MAX_LORAS': '4',                 # adapters resident on GPU simultaneously
    'SM_VLLM_MAX_CPU_LORAS': '8',             # additional adapters cached in host RAM
    'VLLM_ALLOW_RUNTIME_LORA_UPDATING': 'true',
    # Optional, for tool/function calling — the parser is model-family specific:
    # gpt-oss: 'openai'; qwen3: 'hermes'. Uncomment for your base:
    # 'SM_VLLM_ENABLE_AUTO_TOOL_CHOICE': 'true',
    # 'SM_VLLM_TOOL_CALL_PARSER': 'openai',
}
try:
    sm.create_model(ModelName=BASE_MODEL_NAME, ExecutionRoleArn=ROLE_ARN,
        PrimaryContainer={'Image': VLLM_IMAGE, 'Environment': ENV,
            'ModelDataSource': {'S3DataSource': {'S3Uri': BASE_ARTIFACT,
                'S3DataType': 'S3Prefix', 'CompressionType': 'None'}}})
except sm.exceptions.ClientError as e:
    if 'Cannot create already existing model' not in str(e): raise

try:
    sm.create_inference_component(
        InferenceComponentName=BASE_IC, EndpointName=ENDPOINT_NAME, VariantName='main',
        Specification={
            'ModelName': BASE_MODEL_NAME,
            'ComputeResourceRequirements': {
                'NumberOfAcceleratorDevicesRequired': int(TENSOR_PARALLEL),
                'MinMemoryRequiredInMb': 65536},
            'StartupParameters': {'ModelDataDownloadTimeoutInSeconds': 600,
                                  'ContainerStartupHealthCheckTimeoutInSeconds': 600}},
        RuntimeConfig={'CopyCount': 1})
    print('base IC create requested')
except sm.exceptions.ClientError as e:
    if 'already exist' not in str(e).lower(): raise

t0 = time.time()
while True:
    d = sm.describe_inference_component(InferenceComponentName=BASE_IC)
    if d['InferenceComponentStatus'] == 'InService':
        print(f'base IC InService after {time.time()-t0:.0f}s'); break
    if d['InferenceComponentStatus'] == 'Failed':
        raise RuntimeError(f"base IC failed: {d.get('FailureReason')} — see CloudWatch "
                           f'/aws/sagemaker/InferenceComponents/{BASE_IC}')
    print(f"{d['InferenceComponentStatus']} ({time.time()-t0:.0f}s)"); time.sleep(30)

## Step 6 — Attach your fine-tune as an adapter component

Adapter components accept exactly two things: the base component's name and the tarball
location. They use the base's compute, so they attach in **seconds** — this is the step you
repeat for every new fine-tune.


In [ ]:
ADAPTER_IC = f'{ENDPOINT_NAME}-adapter-v{pkg_version}'
try:
    sm.create_inference_component(
        InferenceComponentName=ADAPTER_IC, EndpointName=ENDPOINT_NAME,
        Specification={'BaseInferenceComponentName': BASE_IC,
                       'Container': {'ArtifactUrl': ADAPTER_TARBALL}})
except sm.exceptions.ClientError as e:
    if 'already exist' not in str(e).lower(): raise
while True:
    d = sm.describe_inference_component(InferenceComponentName=ADAPTER_IC)
    if d['InferenceComponentStatus'] in ('InService', 'Failed'):
        print(ADAPTER_IC, '->', d['InferenceComponentStatus']); break
    time.sleep(10)

## Step 7 — Invoke

OpenAI chat-completions JSON. `InferenceComponentName` selects **which model answers**:
the adapter component for your fine-tune, the base component for the raw model —
same endpoint, same request format.


In [ ]:
def chat(ic_name, user_msg, system_msg=None, max_tokens=512, temperature=0.0, stream=False):
    body = {'messages': ([{'role': 'system', 'content': system_msg}] if system_msg else [])
                        + [{'role': 'user', 'content': user_msg}],
            'max_tokens': max_tokens, 'temperature': temperature}
    if ic_name != BASE_IC:
        body['model'] = '/opt/ml/model'      # adapter routing rewrites this to the adapter
    if stream:
        body['stream'] = True
        r = smr.invoke_endpoint_with_response_stream(
            EndpointName=ENDPOINT_NAME, InferenceComponentName=ic_name,
            ContentType='application/json', Body=json.dumps(body))
        for ev in r['Body']:
            for line in ev.get('PayloadPart', {}).get('Bytes', b'').decode(errors='ignore').splitlines():
                if line.startswith('data:') and line != 'data: [DONE]':
                    try:
                        print(json.loads(line[5:])['choices'][0]['delta'].get('content') or '', end='')
                    except Exception: pass
        print(); return
    r = smr.invoke_endpoint(EndpointName=ENDPOINT_NAME, InferenceComponentName=ic_name,
                            ContentType='application/json', Body=json.dumps(body))
    return json.loads(r['Body'].read())['choices'][0]['message']

SYSTEM = "Solve the math problem. End your answer with '#### <number>'."
Q = ('Natalia sold clips to 48 of her friends in April, and then she sold half as many '
     'clips in May. How many clips did Natalia sell altogether in April and May?')

print('--- your fine-tune ---')
print(chat(ADAPTER_IC, Q, SYSTEM)['content'])
print('\n--- base model, same endpoint ---')
print(chat(BASE_IC, Q, SYSTEM)['content'])
print('\n--- streaming ---')
chat(ADAPTER_IC, Q, SYSTEM, stream=True)

## Step 8 — Add more fine-tunes to the same endpoint

This is the payoff of the multi-LoRA shape. For any other model package from your
training runs (a different run, or an earlier checkpoint of the same run), stage its
adapter and attach it — the base keeps serving, other adapters are unaffected, and
each request routes by `InferenceComponentName`.


In [ ]:
# EXAMPLE — uncomment and set another package version:
# OTHER_PACKAGE_ARN = 'arn:aws:sagemaker:...:model-package/<group>/<other-version>'
# ... repeat Step 2's adapter resolution + Step 3's tarball staging for OTHER_PACKAGE_ARN,
# then:
# sm.create_inference_component(
#     InferenceComponentName=f'{ENDPOINT_NAME}-adapter-v<other-version>',
#     EndpointName=ENDPOINT_NAME,
#     Specification={'BaseInferenceComponentName': BASE_IC,
#                    'Container': {'ArtifactUrl': '<other adapter.tar.gz s3 uri>'}})
print('adapters currently on the endpoint:')
for ic in sm.list_inference_components(EndpointNameEquals=ENDPOINT_NAME)['InferenceComponents']:
    print(' ', ic['InferenceComponentName'], '-', ic['InferenceComponentStatus'])

## Step 9 — Clean up

The endpoint bills per instance-hour while `InService`. Deletion order matters:
adapters → base component → endpoint → config → model. (A component stuck in `Creating`
cannot be deleted until it reaches a terminal state — the `StartupParameters` timeouts
above bound that wait.)


In [ ]:
CLEAN_UP = False   # set True to delete everything created by this notebook
if CLEAN_UP:
    for ic in sm.list_inference_components(EndpointNameEquals=ENDPOINT_NAME)['InferenceComponents']:
        name = ic['InferenceComponentName']
        if name != BASE_IC:
            sm.delete_inference_component(InferenceComponentName=name); print('deleting', name)
    def gone(name):
        try: sm.describe_inference_component(InferenceComponentName=name); return False
        except sm.exceptions.ClientError: return True
    while not all(gone(ic['InferenceComponentName'])
                  for ic in sm.list_inference_components(EndpointNameEquals=ENDPOINT_NAME)['InferenceComponents']
                  if ic['InferenceComponentName'] != BASE_IC):
        time.sleep(10)
    sm.delete_inference_component(InferenceComponentName=BASE_IC); print('deleting', BASE_IC)
    while not gone(BASE_IC): time.sleep(15)
    sm.delete_endpoint(EndpointName=ENDPOINT_NAME); print('deleting endpoint')
    sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    sm.delete_model(ModelName=BASE_MODEL_NAME)
    print('done — staged S3 artifacts were kept for future deployments')
else:
    print('set CLEAN_UP = True to tear down (endpoint bills hourly while up)')